# Mini PACS AI - Integração Orthanc → API de Inferência

Este notebook demonstra o fluxo completo:
1. Lista todas as instâncias DICOM armazenadas no Orthanc
2. Baixa cada instância via API REST do Orthanc
3. Envia para a API de inferência (`/predict`)
4. Exibe a imagem e as probabilidades de achados radiológicos

## 1. Imports e Configuração

In [ ]:
import requests
import pydicom
import matplotlib.pyplot as plt
import io
from typing import List, Dict
import pandas as pd

# URLs dos serviços
ORTHANC_URL = "http://localhost:8042"
INFERENCE_API_URL = "http://localhost:8000"

print("✓ Imports realizados com sucesso")

## 2. Funções Auxiliares

In [ ]:
def list_instances() -> List[str]:
    """
    Lista todos os IDs de instâncias DICOM armazenadas no Orthanc.
    """
    response = requests.get(f"{ORTHANC_URL}/instances")
    response.raise_for_status()
    return response.json()


def download_dicom(instance_id: str) -> bytes:
    """
    Baixa o arquivo DICOM bruto de uma instância específica.
    """
    response = requests.get(f"{ORTHANC_URL}/instances/{instance_id}/file")
    response.raise_for_status()
    return response.content


def get_instance_info(instance_id: str) -> Dict:
    """
    Obtém informações sobre uma instância DICOM.
    """
    response = requests.get(f"{ORTHANC_URL}/instances/{instance_id}/simplified-tags")
    response.raise_for_status()
    return response.json()


def call_inference_api(dicom_bytes: bytes) -> Dict:
    """
    Envia arquivo DICOM para a API de inferência e retorna as probabilidades.
    """
    files = {"dicom_file": ("image.dcm", dicom_bytes, "application/dicom")}
    response = requests.post(f"{INFERENCE_API_URL}/predict", files=files)
    response.raise_for_status()
    return response.json()


def display_image_and_predictions(dicom_bytes: bytes, predictions: Dict, title: str = ""):
    """
    Exibe a imagem DICOM e as top 5 probabilidades.
    """
    # Lê a imagem DICOM
    dcm = pydicom.dcmread(io.BytesIO(dicom_bytes))
    pixel_array = dcm.pixel_array

    # Cria figura com 2 subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Plot da imagem
    ax1.imshow(pixel_array, cmap="gray")
    ax1.set_title(title if title else "Imagem DICOM")
    ax1.axis("off")

    # Plot das probabilidades (top 5)
    probs = predictions.get("probabilities", {})
    top_5 = dict(list(probs.items())[:5])

    labels = list(top_5.keys())
    values = list(top_5.values())

    ax2.barh(labels, values, color="steelblue")
    ax2.set_xlabel("Probabilidade")
    ax2.set_title("Top 5 Achados Radiológicos")
    ax2.set_xlim(0, 1)

    # Adiciona valores nas barras
    for i, v in enumerate(values):
        ax2.text(v + 0.01, i, f"{v:.3f}", va="center")

    plt.tight_layout()
    plt.show()

    # Mostra metadados se disponíveis
    if "metadata" in predictions and predictions["metadata"]:
        print("\nMetadados DICOM:")
        for key, value in predictions["metadata"].items():
            print(f"  {key}: {value}")

print("✓ Funções auxiliares definidas")

## 3. Verificar Status dos Serviços

In [ ]:
# Verifica Orthanc
try:
    response = requests.get(f"{ORTHANC_URL}/system")
    response.raise_for_status()
    print("✓ Orthanc está respondendo")
    print(f"  Versão: {response.json().get('Version', 'N/A')}")
except Exception as e:
    print(f"✗ Erro ao conectar com Orthanc: {e}")

# Verifica API de Inferência
try:
    response = requests.get(f"{INFERENCE_API_URL}/health")
    response.raise_for_status()
    data = response.json()
    print("✓ API de Inferência está respondendo")
    print(f"  Status: {data.get('status', 'N/A')}")
    print(f"  Modelo carregado: {data.get('model_loaded', False)}")
except Exception as e:
    print(f"✗ Erro ao conectar com API de Inferência: {e}")

## 4. Listar Instâncias Disponíveis no Orthanc

In [ ]:
instances = list_instances()
print(f"Total de instâncias DICOM no Orthanc: {len(instances)}")

if instances:
    print("\nIDs das instâncias:")
    for i, instance_id in enumerate(instances, 1):
        print(f"  {i}. {instance_id}")
else:
    print("\n⚠️  Nenhuma instância encontrada no Orthanc.")
    print("   Por favor, faça upload de arquivos DICOM via interface web:")
    print(f"   {ORTHANC_URL}")

## 5. Processar Todas as Instâncias

In [ ]:
if not instances:
    print("⚠️  Sem instâncias para processar. Execute as células anteriores.")
else:
    print(f"Processando {len(instances)} instância(s)...\n")

    for idx, instance_id in enumerate(instances, 1):
        print(f"{'='*80}")
        print(f"Instância {idx}/{len(instances)} - ID: {instance_id}")
        print(f"{'='*80}")

        try:
            # Obtém informações da instância
            info = get_instance_info(instance_id)
            patient_id = info.get("PatientID", "N/A")
            study_date = info.get("StudyDate", "N/A")

            title = f"Paciente: {patient_id} | Data: {study_date}"

            # Baixa o DICOM
            print("  → Baixando arquivo DICOM...")
            dicom_bytes = download_dicom(instance_id)
            print(f"    Tamanho: {len(dicom_bytes)} bytes")

            # Chama API de inferência
            print("  → Chamando API de inferência...")
            predictions = call_inference_api(dicom_bytes)
            print("    Inferência concluída")

            # Exibe resultados
            display_image_and_predictions(dicom_bytes, predictions, title)

            print("\n")

        except Exception as e:
            print(f"  ✗ Erro ao processar instância {instance_id}: {e}")
            print("\n")
            continue

    print(f"{'='*80}")
    print("✓ Processamento concluído!")
    print(f"{'='*80}")

## 6. Análise Consolidada (Opcional)

In [ ]:
if instances:
    print("Gerando análise consolidada de todos os exames...\n")

    all_predictions = []

    for instance_id in instances:
        try:
            dicom_bytes = download_dicom(instance_id)
            predictions = call_inference_api(dicom_bytes)
            probs = predictions.get("probabilities", {})

            # Pega top 3 achados
            top_3 = dict(list(probs.items())[:3])
            all_predictions.append({
                "instance_id": instance_id[:8] + "...",
                **top_3
            })
        except Exception as e:
            print(f"Erro ao processar {instance_id}: {e}")

    if all_predictions:
        df = pd.DataFrame(all_predictions)
        print("Top 3 achados por instância:")
        print(df.to_string(index=False))
else:
    print("Sem instâncias para análise consolidada.")

## 7. Testar com Arquivo DICOM Local (Opcional)

In [ ]:
# Descomente e ajuste o caminho se quiser testar com um arquivo local
# LOCAL_DICOM_PATH = "path/to/your/dicom/file.dcm"
#
# with open(LOCAL_DICOM_PATH, "rb") as f:
#     dicom_bytes = f.read()
#
# predictions = call_inference_api(dicom_bytes)
# display_image_and_predictions(dicom_bytes, predictions, "Arquivo Local")

print("Célula de teste local (comentada)")